# Labelling Clusters

#### INPUT  
- cluster_assignments_macro.csv / cluster_assignments_meso.csv / cluster_assignments_micro.csv
- narrative_extractions_normalized.csv
- preprocessed_dataset.tsv

#### WHAT THIS NOTEBOOK DOES
- loads the three-level cluster assignments produced by notebook 05_cluster.ipynb
- runs a small test-labelling preview on a few random clusters per level to sanity-check the prompts
- samples representative examples and top slot fillers for each cluster
- uses the Anthropic API to generate labels and short descriptions at macro, meso and micro level
- merges the three levels into a single labels table

#### OUTPUTS 
- cluster_labels_macro.csv / cluster_labels_meso.csv / cluster_labels_micro.csv --> per-level labels
- cluster_labels.csv --> merged labels across all three levels


In [1]:
# imports

import random
import anthropic
import pandas as pd

from config import label_config as cfg
from lib.label_lib import (
    load_assignments, load_extractions, load_preprocessed, load_labels, save_labels,
    proportional_n_examples, get_examples,
    get_slot_fillers_macro_meso, get_slot_fillers_micro, get_dominant_entities,
    call_llm, label_macro, label_meso, label_micro,
    merge_labels,
)
from prompts.label_macro_prompt import LABEL_PROMPT_MACRO
from prompts.label_meso_prompt import LABEL_PROMPT_MESO
from prompts.label_micro_prompt import LABEL_PROMPT_MICRO

client = anthropic.Anthropic()

In [ ]:
# load data

macro_assignments = load_assignments(cfg.MACRO_ASSIGNMENTS_CSV)
meso_assignments  = load_assignments(cfg.MESO_ASSIGNMENTS_CSV)
micro_assignments = load_assignments(cfg.MICRO_ASSIGNMENTS_CSV)
extractions       = load_extractions(cfg.EXTRACTIONS_CSV)
preprocessed      = load_preprocessed(cfg.PREPROCESSED_CSV)

print(f"Preprocessed articles: {len(preprocessed):,}")
print(f"Macro clusters: {macro_assignments['macro_cluster'].nunique()}")
print(f"Meso clusters:  {meso_assignments['meso_cluster'].nunique()}")
print(f"Micro clusters: {micro_assignments['micro_cluster'].nunique()}")


Preprocessed articles: 64,504
Macro clusters: 17
Meso clusters:  226
Micro clusters: 351


In [6]:
def test_label_cluster(cid, ids, extractions, preprocessed, level, meso_labels_map=None, macro_labels_map=None):
    n_examples = proportional_n_examples(len(ids))
    examples   = get_examples(ids, extractions, preprocessed, n_examples)

    if level == "macro":
        slot_fillers = get_slot_fillers_macro_meso(ids, extractions)
        prompt = LABEL_PROMPT_MACRO.format(
            existing_labels="(none yet)",
            slot_fillers=slot_fillers,
            n_examples=n_examples,
            examples=examples,
        )
    elif level == "meso":
        macro_id        = int(cid) // 1000
        macro_info      = (macro_labels_map or {}).get(macro_id, {})
        slot_fillers    = get_slot_fillers_macro_meso(ids, extractions)
        dominant_ents   = get_dominant_entities(ids, extractions)
        prompt = LABEL_PROMPT_MESO.format(
            parent_title=macro_info.get("title", f"Macro {macro_id}"),
            parent_description=macro_info.get("description", ""),
            existing_labels="(none yet)",
            dominant_entities=dominant_ents,
            slot_fillers=slot_fillers,
            n_examples=n_examples,
            examples=examples,
        )
    else:
        meso_id      = int(cid) // 1000
        macro_id     = meso_id // 1000
        meso_info    = (meso_labels_map or {}).get(meso_id, {})
        macro_info   = (macro_labels_map or {}).get(macro_id, {})
        slot_fillers = get_slot_fillers_micro(ids, extractions)
        prompt = LABEL_PROMPT_MICRO.format(
            macro_title=macro_info.get("title", f"Macro {macro_id}"),
            macro_description=macro_info.get("description", ""),
            meso_title=meso_info.get("title", f"Meso {meso_id}"),
            meso_description=meso_info.get("description", ""),
            existing_labels="(none yet)",
            slot_fillers=slot_fillers,
            n_examples=n_examples,
            examples=examples,
            dominant_entities=get_dominant_entities(ids, extractions)
        )

    sample_titles = (
        preprocessed[preprocessed["id"].isin(ids.astype(str))]["head"]
        .dropna()
        .sample(min(10, preprocessed[preprocessed["id"].isin(ids.astype(str))]["head"].dropna().shape[0]), random_state=42)
        .tolist()
    )
    print(f"\n  Sample titles:")
    for t in sample_titles:
        print(f"    · {t[:80]}")
    result = call_llm(prompt, client)
    print(f"\n  {level.upper()} {int(cid)} (n={len(ids):,})")
    print(f"  Title:       {result.get('title', '')}")
    print(f"  Description: {result.get('description', '')}")
    return result

# --- macro test ---
print("=== Macro (3 samples) ===")
macro_test_ids    = random.sample(list(macro_assignments["macro_cluster"].dropna().unique()), 3)
macro_test_labels = {}
for cid in macro_test_ids:
    ids    = macro_assignments[macro_assignments["macro_cluster"] == cid]["id"].values
    result = test_label_cluster(cid, ids, extractions, preprocessed, "macro")
    macro_test_labels[int(cid)] = result

# --- meso test ---
print("\n=== Meso (3 samples) ===")
meso_test_ids    = random.sample(list(meso_assignments["meso_cluster"].dropna().unique()), 3)
meso_test_labels = {}
for cid in meso_test_ids:
    ids    = meso_assignments[meso_assignments["meso_cluster"] == cid]["id"].values
    result = test_label_cluster(cid, ids, extractions, preprocessed, "meso",
                                macro_labels_map=macro_test_labels)
    meso_test_labels[int(cid)] = result

# --- micro test ---
print("\n=== Micro (3 samples) ===")
meso_label_map  = {k: v for k, v in meso_test_labels.items()}
micro_test_ids  = random.sample(list(micro_assignments["micro_cluster"].dropna().unique()), 3)
for cid in micro_test_ids:
    ids     = micro_assignments[micro_assignments["micro_cluster"] == cid]["id"].values
    meso_id = int(cid) // 1000
    if meso_id not in meso_label_map:
        meso_ids = meso_assignments[meso_assignments["meso_cluster"] == float(meso_id)]["id"].values
        meso_label_map[meso_id] = test_label_cluster(
            float(meso_id), meso_ids, extractions, preprocessed, "meso",
            macro_labels_map=macro_test_labels
        )
    test_label_cluster(cid, ids, extractions, preprocessed, "micro",
                       meso_labels_map=meso_label_map,
                       macro_labels_map=macro_test_labels)

=== Macro (3 samples) ===

  Sample titles:
    · So könnten die USA den Druck auf Russland erhöhen
    · Trump lässt Putins Öltanker nach Kuba
    · Diesen beiden Männern aus Pakistan traut US-Präsident Donald Trump
    · Das internationale Genf steht unter Druck
    · Selenski zu Gespräch mit Putin bereit
    · Tony Blair wird zum Gehilfen des «Friedensstifters» Trump
    · Claudia Sheinbaum managt jede Krise
    · «Die Schweiz wird keine Soldaten nach Gaza schicken»
    · Sie wurde gefoltert – und will nun die UNO führen
    · Karte zeigt: Netanyahu fliegt über eigenartige Route in die USA

  MACRO 0 (n=4,087)
  Title:       Geopolitical Mediation
  Description: Diplomatic actors seek negotiated resolution while rival interests obstruct international settlement.

  Sample titles:
    · Spital erlaubt Eltern, Kaiserschnitt live mitzuverfolgen
    · Acht Monate weg, zwei daheim
    · Hitze fordert Hunderte Schweizer Tote pro Jahr
    · Achtung vor AHV-Beitragslücken!: Sonst gibt es we

In [7]:
# label macro

existing_macro = load_labels(cfg.MACRO_LABELS_CSV)
print(f"Already labelled: {len(existing_macro)} macro clusters")

macro_labels = label_macro(
    assignments     = macro_assignments,
    extractions     = extractions,
    preprocessed    = preprocessed,
    client          = client,
    existing_labels = existing_macro,
)

save_labels(macro_labels, cfg.MACRO_LABELS_CSV)
print(f"\nSaved macro labels → {cfg.MACRO_LABELS_CSV}")

Already labelled: 0 macro clusters
  macro 0: Geopolitical Diplomacy
  macro 1: Military Coercion
  macro 2: Identity Assertion
  macro 3: Law Enforcement
  macro 4: Electoral Rivalry
  macro 5: Media Scrutiny
  macro 6: Institutional Governance
  macro 7: Criminal Adjudication
  macro 8: Criminal Investigation
  macro 9: Political Succession
  macro 10: Authoritarian Resistance
  macro 11: Technological Advancement
  macro 12: Ecological Advocacy
  macro 13: Economic Rivalry
  macro 14: Social Welfare Claims
  macro 15: Policy Reform
  macro 16: Administrative Authority

Saved macro labels → data/cluster_labels_macro.csv


In [8]:
# label meso

existing_meso = load_labels(cfg.MESO_LABELS_CSV)
print(f"Already labelled: {len(existing_meso)} meso clusters")

meso_labels = label_meso(
    assignments     = meso_assignments,
    extractions     = extractions,
    preprocessed    = preprocessed,
    client          = client,
    existing_labels = existing_meso,
)

save_labels(meso_labels, cfg.MESO_LABELS_CSV)
print(f"\nSaved meso labels → {cfg.MESO_LABELS_CSV}")

Already labelled: 0 meso clusters

── Macro 0: Geopolitical Diplomacy ──
  meso 0: Glückskette's War Zone Aid
  meso 1: Trump-Era Peace Plan Clashes

── Macro 1: Military Coercion ──
  meso 1000: Russia-Ukraine War Frontlines
  meso 1001: Trump's Global Force Projection
  meso 1002: Global Military Force Theaters
  meso 1003: Nuclear and Aerospace Security Crises
  meso 1004: Wartime Ceasefire and Sanctions Standoffs
  meso 1005: Switzerland's Military Rearmament Drive
  meso 1006: Multifront Armed Confrontations
  meso 1007: Gulf Crisis Civilian Evacuations
  meso 1008: Western Military Instrument Deployment
  meso 1009: Multilateral Coercive Diplomacy Fronts
  meso 1010: Russia-Ukraine Infrastructure War
  meso 1011: Ukraine-Russia Drone Combat
  meso 1012: Great-Power Strategic Coercion Moves
  meso 1013: Europe's Defence Autonomy Drive
  meso 1014: Global Strategic Force Decisions
  meso 1015: Iran-Israel Direct Military Exchange
  meso 1016: Global Escalation Bargaining Fronts
  m

In [9]:
# label micro

existing_micro = load_labels(cfg.MICRO_LABELS_CSV)
print(f"Already labelled: {len(existing_micro)} micro clusters")

meso_labels_df = pd.read_csv(cfg.MESO_LABELS_CSV)

micro_labels = label_micro(
    micro_assignments = micro_assignments,
    extractions       = extractions,
    preprocessed    = preprocessed,
    meso_labels       = meso_labels_df,
    client            = client,
    existing_labels   = existing_micro,
)

save_labels(micro_labels, cfg.MICRO_LABELS_CSV)
print(f"\nSaved micro labels → {cfg.MICRO_LABELS_CSV}")

Already labelled: 0 micro clusters

── Macro 0: Geopolitical Diplomacy ──
  ── Meso 0: Glückskette's War Zone Aid ──
    micro 0: Swiss authorities mobilising humanitarian aid for war-affected civilians
    micro 1: Glückskette mobilising humanitarian relief for war-affected civilians
    micro 2: Glückskette delivering humanitarian aid amid Gaza crisis
    micro 3: Humanitarian organisations mobilising aid for Gaza civilians amid blockade
    micro 4: Glückskette delivering humanitarian aid to Gaza civilians amid crisis
  ── Meso 1: Trump-Era Peace Plan Clashes ──
    micro 1000: Trump brokering multi-theatre peace deals as self-styled peacemaker
    micro 1001: Foreign governments unilaterally deploying sovereignty-asserting policy instruments
    micro 1002: Trump advancing unilateral geopolitical instruments over international resistance
    micro 1003: Foreign executives negotiating rival diplomatic instruments for conflict resolution
    micro 1004: Foreign executives advancing r

In [10]:
# merge and save labels

macro_labels_df = pd.read_csv(cfg.MACRO_LABELS_CSV)
meso_labels_df  = pd.read_csv(cfg.MESO_LABELS_CSV)
micro_labels_df = pd.read_csv(cfg.MICRO_LABELS_CSV)

all_labels = merge_labels(macro_labels_df, meso_labels_df, micro_labels_df)
all_labels.to_csv(cfg.CLUSTER_LABELS_CSV, index=False)

print(f"Saved merged labels → {cfg.CLUSTER_LABELS_CSV}")
print(f"\nTotal labels: {len(all_labels)}")

Saved merged labels → data/cluster_labels.csv

Total labels: 594
